# Try the personal-card-agent agent

Two ways to try this repo's agent from a notebook:

- **Option A - quick local invoke.** Builds the agent in-process with the
  OSS `deepagents` library, reusing this repo's `instructions.md` and
  `middleware/` (customer-context seeding + tool-call logging). No `mda`
  CLI or server needed - fastest way to iterate.
- **Option B - talk to the real managed agent.** Connects to the agent
  exactly as `agent.py` defines it, served locally by `mda dev .`.
- **Option C - test the data-visualization chart skill.** Builds the agent
  exactly as `server.py` does (same `backend`/`skills` wiring), needs a
  fuller set of env vars than A/B (see that section).
- **Option D - test the transfer-form skill.** Same wiring as Option C, a
  different skill/`ui["type"]` (`transfer_form` -- proposing a paydown for
  the customer's own credit card debt).
- **Option E - test the spending-jar skill.** Same wiring again, a third
  skill/`ui["type"]` pair (`create_jar`/`edit_jar` -- proposing a new or
  changed spending jar).
- **Option J - test `POST /jar-distribute` and `POST /jar-rebalance`.** The two endpoints for screens that need
  no chat (share unallocated money into jars / move balance from one jar to others). Called
  in-process with `TestClient(server.app)`, no conversation involved.
- **Option K - test the jar flows in chat.** The same jar contract through
  `POST /chat`: moving a category between two jars, putting unallocated money into a jar
  (`distribute_amount`), and a jar that does not exist yet (no `ui`).
- **Option L - test the `clarify-options` skill.** Forces a genuinely ambiguous
  situation (two similarly-named jars) so the agent asks with `clarify_options`
  buttons instead of an open question, then simulates the customer tapping one.
- **Option M - test putting an amount into a jar.** "Chuyển 300 nghìn vào hũ Tiết kiệm" straight through the agent
  (`distribute_amount`, skill `distribute-amount`, the money comes from the unallocated pool) -- no endpoint, no `/chat` route.
- **Option N - test moving money from one jar to others.** "Chuyển 500 nghìn từ hũ A sang hũ B" straight through the agent
  (`rebalance_jars` with `from_jar_id` + `allocations`, skill `rebalance-jars`) -- no endpoint, no `/chat` route.


In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Notebook runs from notebooks/, but the agent's code (middleware/, data/,
# instructions.md, .env) lives one level up at the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

import os

assert os.environ.get("AI_PLATFORM_API_KEY"), (
    f"Set AI_PLATFORM_API_KEY in {REPO_ROOT / '.env'} first"
)

from langchain_openai import ChatOpenAI

from text_utils import extract_final_text

# Same GreenNode VNG Cloud MaaS endpoint/model server.py's _default_model()
# uses -- kept in sync manually since this notebook predates that switch and
# Option A/checklist below build their own agent without importing server.py
# (importing it would also require SERVER_API_KEY/MEMORY_ID/
# USER_PREFERENCE_STRATEGY_ID just for a quick local invoke -- see Option C).
LLM_BASE_URL = "https://maas-llm-aiplatform-hcm.api.vngcloud.vn/v1"
LLM_MODEL = os.environ.get("LLM_MODEL") or "z-ai/glm-5.2-hackathon"


def default_model() -> ChatOpenAI:
    return ChatOpenAI(base_url=LLM_BASE_URL, api_key=os.environ["AI_PLATFORM_API_KEY"], model=LLM_MODEL)


In [4]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

## Option A - quick local invoke (no `mda dev` needed)

In [ ]:
from deepagents import create_deep_agent

from middleware.audit import log_tool_calls
from middleware.seed_customer_context import SeedCustomerContext

instructions = (REPO_ROOT / "instructions.md").read_text()

local_agent = create_deep_agent(
    model=default_model(),
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
)

result = local_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Thống kê số tiền tôi đã chi tiêu vào ăn uống"}
        ]
    }
)
print(extract_final_text(result["messages"][-1].content))

## Option B - talk to the managed agent via `mda dev`

In a separate terminal, from the repo root, run:

```bash
mda dev .
```

It prints the local Agent Server URL (default `http://127.0.0.1:2024`) and
the graph/assistant name to use below - it's usually the agent `name` from
`agent.py` (`"personal-card-agent"`), but check the printed Studio URL /
server logs and adjust `ASSISTANT_ID` if it differs.

In [2]:
from langgraph_sdk import get_sync_client

SERVER_URL = "http://127.0.0.1:2024"
ASSISTANT_ID = "personal-card-agent"

client = get_sync_client(url=SERVER_URL)

thread = client.threads.create()
for chunk in client.runs.stream(
    thread["thread_id"],
    ASSISTANT_ID,
    input={
        "messages": [
            {"role": "user", "content": "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?"}
        ]
    },
    stream_mode="values",
):
    if chunk.event == "values":
        print(extract_final_text(chunk.data["messages"][-1]["content"]))

KeyboardInterrupt: 

## Option C - test the data-visualization chart skill

Uses `server.build_agent()` directly (not a fresh `create_deep_agent(...)`
call like Option A) so the run actually exercises the same `backend=
CompositeBackend(...)`/`skills=[...]` wiring `server.py` uses in production --
a plain Option-A-style agent has no `skills=[...]`, so the skill would never
be loaded.

Importing `server` pulls in its full module-level env var checks (not just
`AI_PLATFORM_API_KEY`) -- make sure these are also set in `.env`:
`SERVER_API_KEY`, `MEMORY_ID`, `USER_PREFERENCE_STRATEGY_ID` (the last one
via `tools/memory_tools.py`, imported by `server.py`).

The question below explicitly asks for a `biểu đồ` (chart), echoing the
`data-visualization` skill's own description word-for-word -- the closest
thing to a guaranteed trigger this design allows (the model still decides,
there's no enforced schema). A missing/malformed `ui-json` block just
yields `ui is None`, it never breaks the call.

**Note (2026-09-03):** the response field was renamed from `chart` to `ui`
(a shared envelope for every widget type, discriminated by `ui["type"]`)
-- see `docs/api.md`'s breaking-change note and Option D below for the
second type, `transfer_form`.

In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver, not the real GreenNode Memory checkpointer -- otherwise
# each re-run of this cell keeps appending to the SAME real, persistent
# thread ("test-chart") instead of starting a clean conversation, which
# silently changes the model's answer on later runs (it starts "remembering"
# earlier runs). Re-running this cell always starts a fresh, isolated agent.
chart_agent = server.build_agent(checkpointer=InMemorySaver())

result = chart_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Phân tích và vẽ biểu đồ so sánh chi tiêu các danh mục của tôi tháng này.",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-chart", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

## Option D - test the transfer-form skill (trả nợ thẻ tín dụng)

Same wiring as Option C (`server.build_agent()`), just a different skill
(`skills/project/transfer-form/`) and a different `ui["type"]`
(`"transfer_form"` instead of `"chart"`).

This cell exercises **case (A)** only -- paying down the customer's own
credit card debt (`recipient`/`account_number` schema). See Option F below
for **case (B)**, transferring to a saved beneficiary via the real
`query_beneficiaries()` tool. The model verifies available credit itself
by reading `/customer_data.json` (`read_file`) -- there's no dedicated
tool for case (A), so expect only `read_file` calls in the output, no
tool call.

The question below asks about paying off a specific card's statement --
a direct match for the skill's description, similar to how Option C's
question echoes the chart skill's description.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason: avoids silently
# accumulating real history on GreenNode Memory across notebook re-runs.
transfer_agent = server.build_agent(checkpointer=InMemorySaver())

result = transfer_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tôi nên trả nợ thẻ Visa Signature bao nhiêu?",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-transfer-form", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

## Option E - test the spending-jar skill (tạo/chỉnh sửa hũ chi tiêu)

Same wiring as Option C/D (`server.build_agent()`), a third skill
(`skills/project/spending-jar/`) with two `ui["type"]` values:
`"create_jar"` (a new jar: `category_ids` and `initial_balance` -- the opening balance, a whole number >= 0 -- required, never a `jar_id`) and
`"edit_jar"` (an existing jar, identified by the `jar_id` the model copies from
`get_jar_summary`). Contract: `docs/spending-jar.md`.

The model reads the jars **live from the real `pfm` backend** with
`get_jar_summary()` (`GET /api/jar-summary`) and the customer's own categories
with `get_categories()` (`GET /api/categories`) -- so this cell needs network
access to the deployed `pfm`, like Option F -- and `/customer_data.json` for
`monthly_spending_summary` to ground the proposed amount. `server.py` drops the
`ui` when a `jar_id` or category is not one the model read in that turn
(`ui_guard.py`).

Two questions below, run in the same thread: the first should trigger
`create_jar`, the second `edit_jar` for a jar that already exists in pfm (its
`jar_id` comes from the summary, not from the conversation). The jar names are
live pfm data for `CIF_0001`; adjust question 2 if they have changed.


In [3]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver -- see Option C's comment, same reason. This one matters
# even more here: the 2 questions below are DELIBERATELY run on the same
# thread (to test create_jar then edit_jar in one conversation) -- with the
# real checkpointer, re-running this cell keeps adding a 3rd, 5th, 7th...
# turn to the same real GreenNode thread each time, so "Tạo giúp tôi 1 hũ
# mua sắm" stops being a fresh request after the first run and starts
# correctly answering edit_jar instead (the model remembers offering it
# already) -- confusing if you don't expect it. InMemorySaver here means
# every notebook run is exactly the isolated 2-turn conversation it looks
# like in this cell.
jar_agent = server.build_agent(checkpointer=InMemorySaver())
JAR_THREAD = {"configurable": {"thread_id": "test-spending-jar", "actor_id": "CIF_0001"}}

for question in [
    "Tạo giúp tôi 1 hũ mua sắm hợp lý mỗi tháng.",  # expect create_jar
    "Tăng hạn mức hũ Ăn uống của tôi thêm 200 nghìn được không?",  # expect edit_jar (a jar that exists in pfm; jar_id comes from get_jar_summary)
]:
    result = jar_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=JAR_THREAD,
    )
    answer = extract_final_text(result["messages"][-1].content)
    answer, ui = extract_ui_block(answer)
    print(f"Q: {question}")
    print("ANSWER:\n", answer)
    print("\nUI:\n", ui)
    print("-" * 60)

/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Model call: 6.24s
Calling tool: read_file
Calling tool: get_jar_summary
Calling tool: get_categories
Finished tool: read_file (0.01s)
Finished tool: get_jar_summary (0.26s)
Finished tool: get_categories (0.27s)
Model call: 11.17s
Q: Tạo giúp tôi 1 hũ mua sắm hợp lý mỗi tháng.
ANSWER:
 Danh mục Mua sắm hiện đang thuộc hũ **Hưởng thụ** (cùng với Giải trí).

Chi Mua sắm 3 tháng gần đây: 1.959.000đ (tháng 7) → 2.655.000đ (tháng 8) → 1.661.000đ (tháng 9), trung bình **~2.092.000đ/tháng**.

Mình đề xuất hũ **Mua sắm** riêng biệt hạn mức **2.200.000đ/tháng**, tách khỏi hũ Hưởng thụ để dễ kiểm soát.

Mình nạp sẵn **2.200.000đ** từ tiền chờ phân bổ vào hũ, bạn sửa được số này trên form rồi xác nhận nhé.

UI:
 {'type': 'create_jar', 'jar_name': 'Mua sắm', 'allocation_amount': 2200000, 'initial_balance': 2200000, 'category_ids': ['shopping'], 'reason': 'Chi Mua sắm 3 tháng gần nhất trung bình ~2.092.000đ/tháng; danh mục Mua sắm đang thuộc hũ Hưởng thụ sẽ chuyển sang hũ mới này, hũ Hưởng thụ còn p

## Option F - test the transfer-form skill (chuyển tiền cho người nhận đã lưu)

Same wiring as Option D, testing **case (B)** this time: transferring to a
saved beneficiary. Unlike Option D, this calls the tool
`query_beneficiaries()` (`tools/customer_data_tools.py`), which hits the
**real** `pfm` backend over HTTP (`GET /api/beneficiaries?cif=`, see
`backend_docs/pfm-read-api.md`) -- this cell needs network access to the
deployed `pfm` endpoint, and the beneficiary names below are real data for
`CIF_0001` at the time this was written (re-run `query_beneficiaries()`
directly if the seed data has since changed).

Expect the model to call `query_beneficiaries()` first, find "Lan" in the
result, then propose a `transfer_form` using the new
`beneficiary_id`/`category` schema (NOT `recipient`/`account_number` --
see `docs/transfer-form.md`).


In [4]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver -- see Option C's comment, same reason.
beneficiary_transfer_agent = server.build_agent(checkpointer=InMemorySaver())

result = beneficiary_transfer_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Chuyển tiền cho Lan 500 nghìn với nội dung Thanh toán tiền ăn.",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-transfer-form-beneficiary", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print("ANSWER:\n", answer)
print("\nUI:\n", ui)
assert ui is not None and ui["type"] == "transfer_form"
assert "beneficiary_id" in ui, "expected the pfm-lookup schema (beneficiary_id/category), not recipient/account_number"
assert ui["category"] in {
    "housing", "utilities", "subscriptions", "insurance", "dining",
    "transport", "shopping", "groceries", "entertainment", "health",
}


Model call: 5.76s
Calling tool: query_beneficiaries
Finished tool: query_beneficiaries (0.21s)
Model call: 3.48s
Calling tool: read_file
Finished tool: read_file (0.00s)
Model call: 6.77s
ANSWER:
 Mình đã điền sẵn form chuyển 500.000đ cho Nguyễn Thị Lan, bạn xem lại rồi bấm xác nhận nhé.

UI:
 {'type': 'transfer_form', 'beneficiary_id': 'b_stable_lan', 'amount': 500000, 'note': 'Thanh toan tien an', 'category': 'dining'}


## Option G - test the investment-guidance skill (short answer, decide-then-look-up)

Same wiring as Option D/F (`server.build_agent()`). Tests
`skills/project/investment-guidance/`: a goal-based investment question must get a
**4-5 sentence prose answer** (no tables/headings), grounded in the customer's real
income/spending/card debt, with any scenario folded into the conclusion ("Ngoài ra
bạn có thể cân nhắc ... nhưng cần khoảng ... vốn"). The flow is **decide first,
look up only when needed**: if the advice is "pay the card debt first" there is NO
MSB-product or news lookup; when a lookup is needed it is 1 search + 1 read, and
`middleware/tool_limits.py` hard-caps the calls per request. News is only used if
published within the last 4 days (older articles are refused by the tool); MSB pages
are fetched fresh (10-minute cache). Plain market-news questions are declined.

**Needs `TINYFISH_API_KEY` in `.env`** for the lookups.

Questions (fresh threads): plain news question -> declined; goal question with debt ->
no lookups; goal question as a customer with no jars; "which bank stock" -> at most
1 news search + 1 read.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason.
market_agent = server.build_agent(checkpointer=InMemorySaver())

for i, (actor_id, question) in enumerate(
    [
        ("CIF_0001", "Hôm nay chứng khoán thế nào, vì sao VN-Index giảm?"),  # plain news question -> declined/redirected
        ("CIF_0001", "Tôi muốn tăng tiền tiết kiệm thêm 10% đến cuối năm, tôi nên đầu tư gì?"),  # investment-guidance
        ("CIF_0001", "VN-Index giảm rồi, tôi nên mua cổ phiếu ngân hàng nào?"),  # no "buy X now"
        ("CIF_0002", "Tôi muốn tăng tiền tiết kiệm thêm 10% đến cuối năm, tôi nên đầu tư gì?"),  # own numbers
    ]
):
    result = market_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": f"test-market-{i}", "actor_id": actor_id}},
    )
    print(f"[{actor_id}] Q: {question}")
    print("ANSWER:\n", extract_final_text(result["messages"][-1].content))
    print("-" * 60)


## Option H - test MSB product answers (offline snapshot, with sources)

MSB product questions are answered from an OFFLINE snapshot of MSB's website and
text-layer PDFs (`data/msb_knowledge/`, built by `scripts/sync_msb_knowledge.py`), via ONE
local call to `search_msb_knowledge` (milliseconds, no network). Every passage carries its
source URL and fetch date, so the answer must end with a citation ("Nguồn: URL, dữ liệu lưu
ngày ..."). The old live web-lookup tools were removed and there is no separate skill: the rules live in `instructions.md`, so no extra model round-trip is spent reading a skill.

Expect: 1 tool call (plus the model round-trips), a 2-4 sentence answer, a source URL and
the snapshot date; deposit-rate questions say the rate table is not in the data and point to
MSB's rate page/app; refresh the snapshot with
`uv run python scripts/sync_msb_knowledge.py` (the tool warns when it is older than
`MSB_KB_MAX_AGE_DAYS`, default 7).


In [ ]:
import time

from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason.
msb_agent = server.build_agent(checkpointer=InMemorySaver())

for i, (actor_id, question) in enumerate(
    [
        ("CIF_0001", "MSB có thẻ tín dụng nào hoàn tiền không?"),
        ("CIF_0002", "Tiết kiệm Ong vàng của MSB là gì?"),
        ("CIF_0001", "Phí thường niên thẻ ghi nợ MSB là bao nhiêu?"),  # from the fee PDF
        ("CIF_0003", "Lãi suất tiết kiệm 12 tháng của MSB là bao nhiêu?"),  # rate table not in data -> link, no number
    ]
):
    started = time.time()
    result = msb_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": f"test-msb-{i}", "actor_id": actor_id}},
    )
    print(f"[{actor_id}] Q: {question}  ({time.time() - started:.1f}s)")
    print("ANSWER:\n", extract_final_text(result["messages"][-1].content))
    print("-" * 60)


## Option I - test the investment scenario projection (+ chart)

Tool `project_investment_growth` (`tools/investment_tools.py`, pure arithmetic)
computes, under a HYPOTHETICAL monthly return (default +2%/month -- a
product-owner assumption, not a measured market average): the capital needed to
earn a target gain by 31/12, and the value of a chosen capital at each
month-end for three scenarios (+r, flat 0%, -r). It also returns a ready-made
`ui-json` line chart. The agent must call it a "kịch bản giả định", always show
the flat and losing scenarios, and NOT assume the customer has a jar named
"tiết kiệm" -- capital comes from an amount the customer states, or is
estimated from balances (account - emergency fund - card debt; the money in
the jars is already part of the account balance, so it is not added), with 50% of the idle money proposed by default.

Answers are 4-5 sentences (charts only if the customer asks). Four questions, fresh threads: (1) target only, no stated amount; (2) a stated
amount + target; (3) another customer with no jars; (4) a large stated amount
(target reachable only in the +r scenario). Chart blocks are split out by
`extract_ui_block`, so `ui` prints the validated chart.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason.
scenario_agent = server.build_agent(checkpointer=InMemorySaver())

for i, (actor_id, question) in enumerate(
    [
        ("CIF_0001", " "),
        # ("CIF_0002", "Tôi có 60 triệu nhàn rỗi, muốn có thêm 10 triệu đến cuối năm, tôi nên đầu tư gì?"),
        # ("CIF_0003", "Tôi muốn có thêm 10 triệu đến cuối năm, tôi nên đầu tư gì?"),
        # ("CIF_0003", "Tôi có 400 triệu nhàn rỗi, muốn có thêm 10 triệu đến cuối năm, nên đầu tư thế nào?"),
    ]
):
    result = scenario_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": f"test-scenario-{i}", "actor_id": actor_id}},
    )
    answer, ui = extract_ui_block(extract_final_text(result["messages"][-1].content))
    print(f"[{actor_id}] Q: {question}")
    print("ANSWER:\n", answer)
    print("UI (chart):\n", ui)
    print("-" * 60)


Model call: 6.63s
Calling tool: read_file
Finished tool: read_file (0.00s)
Model call: 5.26s
Calling tool: search_msb_products
Calling tool: search_msb_products
Calling tool: search_msb_products
Finished tool: search_msb_products (1.23s)
Finished tool: search_msb_products (1.57s)
Finished tool: search_msb_products (1.87s)
Model call: 6.48s
Calling tool: search_msb_products
Calling tool: read_msb_product_page
Calling tool: read_msb_product_page
Finished tool: search_msb_products (1.25s)
Finished tool: read_msb_product_page (3.96s)
Finished tool: read_msb_product_page (4.82s)
Model call: 5.11s
Calling tool: search_msb_products
Calling tool: search_msb_products
Finished tool: search_msb_products (1.26s)
Finished tool: search_msb_products (1.26s)
Model call: 5.27s
Calling tool: read_msb_product_page
Finished tool: read_msb_product_page (2.82s)
Model call: 20.76s
[CIF_0001] Q: MSB Có sản phẩm gì để đầu tư không
ANSWER:
 MSB có 3 nhóm sản phẩm để bạn đầu tư/sinh lời:

**1. Tiết kiệm & tiền g

## Option J - test `POST /jar-distribute` and `POST /jar-rebalance` (không cần chat)

The two endpoints for screens that already know what to do, so there is no conversation
(`jar_endpoint.py`; contract, error codes and what the agent is handed in `docs/jars/endpoints.md`).
Called in-process with `TestClient(server.app)` -- the same wiring `/chat` lives in:

- **`POST /jar-distribute`** `{"user_id", "amount"}`: share an amount of the customer's **unallocated money** into the jars'
  balances (never the limits) -> a `distribute_amount` `ui`. The agent decides which jars and how much.
- **`POST /jar-rebalance`** `{"user_id", "from_jar_id", "amount"}`: move an amount of one jar's balance to one or more other
  jars -> a `rebalance_jars` `ui` (`from_jar_id` + `allocations`).

For both, the server reads the jar summary **and the last 3 months of spending per jar** and hands them to the
model, so the split follows real need. The cell reads the real `jar-summary` from `pfm` first (so it picks a real jar
and survives data changes), calls each endpoint, then runs the SAME deterministic checks the server uses
(`jar_checks.check_distribute` / `check_rebalance`) and asserts no hard rule is broken. It needs network access to `pfm`
and to the model, and `SERVER_API_KEY` in `../.env` (the endpoints use the same `X-API-Key` as `/chat`). Each call takes
~10-40 seconds. When the pool or the jar cannot cover the amount the server answers by itself with `ui: null` -- that is a
valid result too, the cell prints it.

In [5]:
import os

from fastapi.testclient import TestClient

import server
from jar_checks import check_distribute, check_rebalance, pool_available, source_available
from tools.customer_data_tools import fetch_jar_summary

assert os.environ.get("SERVER_API_KEY"), f"Set SERVER_API_KEY in {REPO_ROOT / '.env'} first"

client = TestClient(server.app)
HEADERS = {"X-API-Key": os.environ["SERVER_API_KEY"]}
CUSTOMER = "CIF_0001"

# The real jar-summary decides which jar to use, so this cell keeps working when the pfm data changes.
summary = fetch_jar_summary(CUSTOMER)
print("jars (limit, balance):", {j["label"]: (j["limit"], j["balance"]) for j in summary["jars"]})
print("unallocated:", summary["unallocated"])
print("=" * 60)

# --- POST /jar-distribute: share unallocated money into the jars (the agent picks the jars) ---
AMOUNT = 500_000
response = client.post("/jar-distribute", headers=HEADERS, json={"user_id": CUSTOMER, "amount": AMOUNT})
assert response.status_code == 200, response.text
body = response.json()
print(f"DISTRIBUTE {AMOUNT} (unallocated now: {pool_available(summary)})")
print("ANSWER:\n", body["answer"])
print("\nUI:\n", body["ui"])
if body["ui"] is not None:
    verdict = check_distribute(summary, AMOUNT, body["ui"])
    assert verdict.hard == [], verdict.hard
    print("soft notes (only wording / money left undistributed):", verdict.soft)
print("=" * 60)

# --- POST /jar-rebalance: move balance from the jar that holds the most into other jars ---
giver = max((j for j in summary["jars"] if j.get("balance") is not None), key=lambda j: j["spendable"] or 0, default=None)
if giver is None:
    print("No jar has a balance yet, so there is nothing to move -- skipping /jar-rebalance.")
else:
    amount = min(500_000, source_available(summary, giver["id"]))
    response = client.post(
        "/jar-rebalance",
        headers=HEADERS,
        json={"user_id": CUSTOMER, "from_jar_id": giver["id"], "amount": amount},
    )
    assert response.status_code == 200, response.text
    body = response.json()
    print(f"REBALANCE {amount} from {giver['label']} (spendable {giver['spendable']})")
    print("ANSWER:\n", body["answer"])
    print("\nUI:\n", body["ui"])
    if body["ui"] is not None:
        verdict = check_rebalance(summary, giver["id"], amount, body["ui"])
        assert verdict.hard == [], verdict.hard
        print("soft notes (only wording / money left in the giver):", verdict.soft)

jars (budgetLimit, remaining): {'Thiết yếu': (8000000, 51000), 'Ăn uống': (4000000, 1813000), 'Di chuyển': (1500000, 796000), 'Hưởng thụ': (2500000, 1296000), 'Sức khỏe': (1000000, 1000000), 'Tiết kiệm': (None, None)}
unallocated: 13044000


distribute-amount: attempt 1 not clean: hard=[] soft=['reason không được nhắc tên nội bộ (shortfall); dùng cách nói thường của khách.']
distribute-amount: model call failed (attempt 2)
Traceback (most recent call last):
  File "/media/daonguyen/Dual/MSB/MSBHackathon/personal-card-agent/jar_endpoint.py", line 170, in _run
    draft = _propose(task.prompt, task.schema, messages)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/media/daonguyen/Dual/MSB/MSBHackathon/personal-card-agent/jar_endpoint.py", line 143, in _propose
    return agent.invoke({"messages": messages})["structured_response"]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/main.py", line 3365, in invoke
    for chunk in self.stream(
                 ^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/main.py", line 2759, in stream
    for _ in runner.tick(
           

COVER Thiết yếu: spend 1051000, shortfall 1000000
ANSWER:
 Hũ Thiết yếu đang thiếu 1.000.000đ cho khoản chi 1.051.000đ.

Đề xuất lấy đủ 1.000.000đ từ tiền chưa phân bổ — không cần động đến hũ nào khác.

UI:
 {'type': 'rebalance_jars', 'target_jar_id': 'essentials', 'shortfall': 1000000, 'moves': [{'from_jar_id': 'pool', 'amount': 1000000}], 'reason': 'Tiền chưa phân bổ (13.044.000đ) đủ để bù trọn 1.000.000đ shortfall của hũ Thiết yếu, nên không cần lấy từ hũ khác.'}
soft notes (only the preferred order/wording): []


distribute-amount: model call failed (attempt 1)
Traceback (most recent call last):
  File "/media/daonguyen/Dual/MSB/MSBHackathon/personal-card-agent/jar_endpoint.py", line 170, in _run
    draft = _propose(task.prompt, task.schema, messages)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/media/daonguyen/Dual/MSB/MSBHackathon/personal-card-agent/jar_endpoint.py", line 143, in _propose
    return agent.invoke({"messages": messages})["structured_response"]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/main.py", line 3365, in invoke
    for chunk in self.stream(
                 ^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/main.py", line 2759, in stream
    for _ in runner.tick(
             ^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/_runner.py", line 167, in tick
 

AssertionError: {"detail":"The model is unavailable, try again shortly"}

## Option K - test the jar flows in chat (`POST /chat`)

The chat side of the jar contract (`docs/jars/`), run
through the real `POST /chat` route with `TestClient(server.app)` on an in-memory agent
(`server.agent` is swapped for `server.build_agent(checkpointer=InMemorySaver())`, so every run
starts a fresh thread -- see Option C). Going through the route means the shape validators
(`text_utils.py`) **and** `ui_guard.py` run, exactly as they do for the frontend. The model
reads the live jars from `pfm` (`get_jar_summary`, `get_categories`), so the cell needs network
access to `pfm` and to the model, and the questions are built from the real jars (they keep working
when the data changes). Four calls, ~15-40 seconds each:

1. **Category move** -- "chuyển danh mục X từ hũ A sang hũ B". Expect ONE `edit_jar` for the
   receiving jar B with B's full category list plus X (an `edit_jar` for A would leave X in no jar).
2. **Put unallocated money into a jar** (skill `distribute-amount`) -- "chuyển 300 nghìn vào hũ X". Expect
   `distribute_amount` with that jar and the amount, and the SAME deterministic checks the server's endpoint uses
   (`jar_checks.check_distribute`) must find no hard error. The money comes only from the unallocated pool
   (`unallocated`), never from another jar; when the pool is empty the model should say so and return no `ui`.
3. **A jar that only exists in the conversation** -- ask for a new jar "Liên thiên", then ask to
   raise it in the next turn. The first turn may propose `create_jar`, but the customer has not
   confirmed it, so the jar is not in `get_jar_summary`: expect **no `ui`** on the second turn (the
   model must not invent a `jar_id`, and `ui_guard.py` drops the form if it does).


In [4]:
import os

import httpx
from fastapi.testclient import TestClient
from langgraph.checkpoint.memory import InMemorySaver

import server
from jar_checks import check_distribute, pool_available
from tools.customer_data_tools import PFM_BASE_URL, fetch_jar_summary

assert os.environ.get("SERVER_API_KEY"), f"Set SERVER_API_KEY in {REPO_ROOT / '.env'} first"

# In-memory agent: every run of this cell is a fresh thread (see Option C for why).
server.agent = server.build_agent(checkpointer=InMemorySaver())
client = TestClient(server.app)
HEADERS = {"X-API-Key": os.environ["SERVER_API_KEY"]}
CUSTOMER = "CIF_0001"


def chat(message: str, thread: str) -> dict:
    response = client.post("/chat", headers=HEADERS, json={"message": message, "user_id": CUSTOMER, "thread_id": thread})
    assert response.status_code == 200, response.text
    body = response.json()
    print(f"Q: {message}\nANSWER:\n {body['answer']}\n\nUI:\n {body['ui']}\n" + "-" * 60)
    return body


def vnd(amount: int) -> str:
    return f"{amount:,}".replace(",", ".") + "đ"


# The real jars decide the questions, so this cell keeps working when the pfm data changes.
summary = fetch_jar_summary(CUSTOMER)
categories = httpx.get(f"{PFM_BASE_URL}/api/categories", params={"cif": CUSTOMER}, timeout=10).json()
with_limit = [j for j in summary["jars"] if j.get("limit") is not None]

# --- 1. move a category from jar A to jar B: ONE edit_jar, for the RECEIVING jar B ---
receiver = with_limit[0]
giver = next(j for j in summary["jars"] if j["id"] != receiver["id"] and j["categoryIds"])
moved = giver["categoryIds"][0]
moved_label = next(c["label"] for c in categories if c["id"] == moved)
ui = chat(f"Chuyển danh mục {moved_label} từ hũ {giver['label']} sang hũ {receiver['label']}.", "k-move")["ui"]
assert ui is not None and ui["type"] == "edit_jar", "expected an edit_jar"
assert ui["jar_id"] == receiver["id"], "must edit the RECEIVING jar, not the giving one"
assert set(ui["category_ids"]) == set(receiver["categoryIds"]) | {moved}, "the receiving jar's full list + the moved category"

# --- 2. put unallocated money into a jar (skill distribute-amount): same contract and same checks as POST /jar-distribute ---
target = with_limit[0]
amount = min(300_000, pool_available(summary)) or 300_000
ui = chat(f"Chuyển {vnd(amount)} vào hũ {target['label']} giúp tôi.", "k-topup")["ui"]
if ui is None:
    print("(no ui: the unallocated money is not enough, or the model explained instead -- valid)")
else:
    assert ui["type"] == "distribute_amount", ui["type"]
    verdict = check_distribute(summary, amount, ui)
    assert verdict.hard == [], verdict.hard
    print("soft notes (only wording / leaving some undistributed):", verdict.soft)

# --- 3. a jar that only exists in the conversation cannot be edited: no ui ---
chat("Tạo giúp tôi hũ Liên thiên 500 nghìn.", "k-ghost")  # may propose create_jar; the customer never confirms it
ui = chat("Tăng hũ Liên thiên lên 700 nghìn giúp tôi.", "k-ghost")["ui"]
assert ui is None, "an edit_jar for a jar that is not in get_jar_summary must never reach the frontend"


/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Model call: 7.95s
Calling tool: read_file
Calling tool: get_jar_summary
Finished tool: read_file (0.00s)
Finished tool: get_jar_summary (0.21s)
Model call: 6.15s
Calling tool: get_categories
Finished tool: get_categories (0.22s)
Model call: 10.78s
Q: Chuyển danh mục Ăn uống từ hũ Ăn uống sang hũ Thiết yếu.
ANSWER:
 Danh mục Ăn uống sẽ chuyển từ hũ Ăn uống sang hũ Thiết yếu.

Hũ Ăn uống còn phụ trách Nhu yếu phẩm, hạn mức hai hũ giữ nguyên, bạn xem lại rồi xác nhận nhé.

UI:
 {'type': 'edit_jar', 'jar_id': 'essentials', 'jar_name': 'Thiết yếu', 'allocation_amount': 9600000, 'category_ids': ['housing', 'utilities', 'insurance', 'subscriptions', 'dining'], 'reason': 'Chuyển danh mục Ăn uống từ hũ Ăn uống sang hũ Thiết yếu; hũ Ăn uống còn phụ trách Nhu yếu phẩm, hạn mức hai hũ giữ nguyên'}
------------------------------------------------------------
Model call: 3.83s
Calling tool: get_jar_summary
Calling tool: read_file
Finished tool: read_file (0.00s)
Finished tool: get_jar_summary (0.22s

## Option L - test the `clarify-options` skill (`clarify_options`)

`clarify_options` (`skills/project/clarify-options/SKILL.md`, contract in
`docs/clarify-options.md`) is a cross-cutting widget: instead of an open-ended
follow-up question, the agent offers 1-4 concrete choices as buttons for a
question it genuinely cannot decide on its own. It is a **last resort** --
so it needs a real ambiguity to trigger, not a stubbed one.

`CIF_0001` has 6 real jars (Thiet yeu/An uong/Di chuyen/Huong thu/Suc
khoe/Tiet kiem) -- more than the schema's 4-option cap, so the skill's own
"KHONG DUOC liet ke qua 4 phuong an" rule forbids offering all 6 as buttons
for a fully unscoped "which jar" question (that must fall back to text --
not a bug, that is what an earlier run of this cell showed). To get a real,
non-stubbed jar-selection `clarify_options` reliably, this cell states the
amount up front (so that is not part of the ambiguity -- see below) and
names the jar with a description that does not literally match any of the
6 real labels ("hu chi tieu hang ngay"). The agent cannot resolve that
against the real jars it just read, so it must ask **which real jar** the
customer meant -- offering up to 4 of them (real names + real
`budgetLimit`) as buttons, per the schema's own cap.

Same wiring as Options C-I (`server.build_agent()` + `agent.invoke()`
directly, `InMemorySaver` -- see Option C's comment for why), not the HTTP
endpoint style of Options J/K. Two turns, same thread:
1. Ask "Tang han muc hu chi tieu hang ngay cua toi len 300.000d giup toi."
   -- amount is already stated (300.000d), so the only unstated thing is
   *which* real jar. Expect `ui["type"] == "clarify_options"` with **one**
   question and up to 4 real jars (name + budgetLimit) as options.
2. Simulate the customer tapping the first option -- send its exact string
   back as the next chat message (the whole reply mechanism for a single
   question: no structured payload, just the button's text as a normal
   message). Expect the agent to now propose `edit_jar` for that jar, with
   its limit raised by 300.000d, having re-read `get_jar_summary` this turn.

**Verified with 5/5 real runs** against `CIF_0001`'s live data: every run
produced a clean single-question `clarify_options` with exactly 4 real jars
as options (the model drops 2 of the 6 each time -- which 2 varies, but the
cap and the real names/amounts held every time). Still printed rather than
asserted, since the model is non-deterministic and could occasionally do
something else (e.g. ask by plain text instead) -- if that happens, rerun
the cell.


In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver -- see Option C's comment, same reason.
clarify_agent = server.build_agent(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "test-clarify-options", "actor_id": "CIF_0001"}}

# Amount is stated (300.000d) so it is not part of the ambiguity. "hu chi tieu hang ngay" matches
# none of CIF_0001's 6 real jars (Thiet yeu/An uong/Di chuyen/Huong thu/Suc khoe/Tiet kiem) -- the
# agent must ask which real one was meant, capped at 4 options per the schema.
question = "Hỏi tôi các câu hỏi để biết nhu cầu đầu tư của tôi."
result = clarify_agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print(f"Q: {question}")
print("ANSWER:\n", answer)
print("\nUI:\n", ui)
print("-" * 60)

if ui is not None and ui.get("type") == "clarify_options":
    # Only one question this time -- tap its first button, send that exact string as the next message.
    pick = ui["questions"][0]["options"][0]
    result2 = clarify_agent.invoke({"messages": [{"role": "user", "content": pick}]}, config=config)
    answer2 = extract_final_text(result2["messages"][-1].content)
    answer2, ui2 = extract_ui_block(answer2)
    print(f"Q (đã chọn): {pick}")
    print("ANSWER:\n", answer2)
    print("\nUI:\n", ui2)
else:
    print("Lần này model không dùng clarify_options (không tất định, xem ghi chú ở trên) -- chạy lại ô này để thử lại.")


Model call: 7.48s
Calling tool: read_file
Finished tool: read_file (0.00s)
Model call: 9.24s
Q: Hỏi tôi các câu hỏi để biết nhu cầu đầu tư của tôi.
ANSWER:
 Để tư vấn đầu tư phù hợp, tôi cần hiểu thêm về bạn. Tôi sẽ hỏi từng bước nhé.

**Bạn đang có số tiền nào dự định đầu tư?** (ví dụ 30 triệu, 60 triệu...)

**Mục tiêu của bạn khi đầu tư là gì?** — tích lũy dài hạn, sinh thêm thu nhập hàng tháng, hay đạt một số tiền cụ thể vào thời điểm nào?

**Bạn có thể chấp nhận rủi ro ở mức nào?** — không muốn mất vốn, chấp nhận biến động nhẹ, hay sẵn sàng rủi ro cao để kỳ vọng lợi nhuận lớn hơn?

**Bạn dự kiến giữ khoản đầu tư trong bao lâu?** — dưới 6 tháng, 1–2 năm, hay trên 3 năm?

Bạn trả lời những câu trên, tôi sẽ tư vấn dựa trên tình hình tài chính thật của bạn.

UI:
 None
------------------------------------------------------------
Lần này model không dùng clarify_options (không tất định, xem ghi chú ở trên) -- chạy lại ô này để thử lại.


## Option M - test putting an amount into a jar (`distribute_amount`, agent called directly)

"Chuyển 300 nghìn vào hũ Tiết kiệm" puts a specific amount of **unallocated money** (tiền chưa phân bổ, "Chờ phân bổ") into a jar's balance -- the job of
the `distribute_amount` form (skill `skills/project/distribute-amount/`, contract `docs/jars/distribute-amount.md`).
The money comes only from the unallocated pool (`unallocated`) and is **never taken from another jar** (không lấy từ hũ khác). Several jars in one sentence
("chia 1 triệu vào Ăn uống và Di chuyển") go into ONE form. A jar with no balance yet (like Tiết kiệm) can still receive: the deposit is
its first balance. The agent only **proposes**: nothing is moved here -- `pfm` writes it when the customer confirms.

Same wiring as Options C-I and L: `server.build_agent()` + `agent.invoke()` directly, `InMemorySaver` (see Option C's comment). No
`TestClient`, no `POST /chat`, no `POST /jar-distribute`, so this is the model + skill + tools only. It reads the real jars from `pfm`
(`get_jar_summary`), so it needs network access to `pfm` and the model.

The cell prints the jar and the unallocated money BEFORE asking: if the pool holds less than the amount, the right answer is **no `ui`**
and a sentence with the real figure. When a form comes back, the cell runs the same deterministic check the endpoint uses
(`jar_checks.check_distribute`): `hard` must be `[]`; `soft` is only wording or money left undistributed.

Not asserted: the model is non-deterministic, so the cell prints what happened instead of failing on an unlucky run.

**Known gap:** `pfm`'s frontend has no card for `distribute_amount` yet (its `agent-api.ts` knows chart, transfer_form, create_jar, edit_jar,
rebalance_jars, clarify_options), so a real UI would show only `answer` until `pfm` adds it.


In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from jar_checks import check_distribute
from text_utils import extract_final_text, extract_ui_block
from tools.customer_data_tools import fetch_jar_summary

# InMemorySaver -- see Option C's comment, same reason.
topup_agent = server.build_agent(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "test-jar-topup", "actor_id": "CIF_0001"}}

# What the jar and the pool really hold right now (the same live pfm data the agent reads). The money comes from `unallocated`;
# `balance: None` = the jar has no balance yet, which does not stop it from receiving.
summary = fetch_jar_summary("CIF_0001")
jar = next((j for j in summary["jars"] if j["label"] == "Tiết kiệm"), None)
print("Tiết kiệm ->", None if jar is None else {k: jar.get(k) for k in ("limit", "balance", "spendable")})
print("unallocated:", summary["unallocated"])
print("=" * 60)

question = "Chuyển 300 nghìn vào hũ Tiết kiệm giúp tôi."
result = topup_agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print(f"Q: {question}")
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

if ui is not None and ui.get("type") == "distribute_amount":
    verdict = check_distribute(summary, 300_000, ui)
    print("\ncheck_distribute -- hard errors (must be []):", verdict.hard)
    print("check_distribute -- soft notes:", verdict.soft)
else:
    print("\nKhông có distribute_amount lần này -- hợp lệ nếu tiền chưa phân bổ không đủ 300.000đ (xem unallocated ở trên).")


## Option N - test moving money from one jar to others (`rebalance_jars`, agent called directly)

"Chuyển 500 nghìn từ hũ Ăn uống sang hũ Di chuyển" moves balance **out of a jar** into one or more other jars -- the job of the
`rebalance_jars` form (skill `skills/project/rebalance-jars/`, contract `docs/jars/rebalance-jars.md`): ONE giving jar
(`from_jar_id`) and one or more receivers (`allocations`, each `to_jar_id` + `amount`); the total can be at most what the giver
holds (`spendable`) and the giver never receives. The unallocated pool is not involved here -- that is Option M
(`distribute-amount`). The agent only **proposes**: nothing is moved -- `pfm` writes it when the customer confirms.

Same wiring as Option M: `server.build_agent()` + `agent.invoke()` directly, `InMemorySaver`. No `TestClient`, no `POST /chat`.
The question is built from the real jars (`fetch_jar_summary`): the jar that holds the most gives to another jar, so the cell
survives data changes. When a form comes back, the cell runs `jar_checks.check_rebalance`: `hard` must be `[]`.

Not asserted: the model is non-deterministic, so the cell prints what happened instead of failing on an unlucky run.

**Known gap:** `pfm`'s `AgentRebalanceCard` still expects the old shape (`target_jar_id`/`shortfall`/`moves`); it has to move to `from_jar_id` +
`allocations`.

In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from jar_checks import check_rebalance
from text_utils import extract_final_text, extract_ui_block
from tools.customer_data_tools import fetch_jar_summary

# InMemorySaver -- see Option C's comment, same reason.
rebalance_agent = server.build_agent(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "test-jar-rebalance", "actor_id": "CIF_0001"}}

# The jar that holds the most gives to another jar; the money comes from the giver, never from `unallocated`.
summary = fetch_jar_summary("CIF_0001")
funded = [j for j in summary["jars"] if j.get("balance") is not None]
giver = max(funded, key=lambda j: j["spendable"] or 0)
receiver = next(j for j in summary["jars"] if j["id"] != giver["id"])
print("giver   ->", {k: giver.get(k) for k in ("label", "limit", "balance", "spendable")})
print("receiver ->", {k: receiver.get(k) for k in ("label", "limit", "balance", "spendable")})
print("=" * 60)

AMOUNT = min(500_000, giver["spendable"] or 0)
question = f"Chuyển {AMOUNT:,} từ hũ {giver['label']} sang hũ {receiver['label']} giúp tôi.".replace(",", ".")
result = rebalance_agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print(f"Q: {question}")
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

if ui is not None and ui.get("type") == "rebalance_jars":
    verdict = check_rebalance(summary, giver["id"], AMOUNT, ui)
    print("\ncheck_rebalance -- hard errors (must be []):", verdict.hard)
    print("check_rebalance -- soft notes:", verdict.soft)
else:
    print("\nKhông có rebalance_jars lần này -- hợp lệ nếu hũ cho không đủ số dư (xem giver ở trên).")

## Verification checklist (local, rows 1-9 and 11 only — row 10 needs Studio, see docs/superpowers/plans/2026-08-30-personal-card-agent-verification.md)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

verify_agent = create_deep_agent(
    model=default_model(),
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
    checkpointer=MemorySaver(),
)

CHECKLIST_QUESTIONS = [
    "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?",
    "Giao dịch lớn nhất tuần vừa rồi là gì?",
    "Liệt kê các giao dịch trên 2 triệu trong tháng 7",
    'Danh mục nào đang "ngốn" tiền của tôi nhất?',
    "Tôi có đang trả lãi thẻ tín dụng không, và vì sao?",
    "Tôi nên dùng thẻ tín dụng hay ghi nợ cho khoản chi 5 triệu này?",
    "Ngày sao kê và ngày đến hạn thanh toán của thẻ tôi là khi nào?",
    "Tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Ngân sách của tôi là 15 triệu/tháng.",
    "Vậy tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Khoá thẻ của tôi giúp tôi.",
]
THREAD_CONFIG = {"configurable": {"thread_id": "notebook-verify-1"}}

for question in CHECKLIST_QUESTIONS:
    result = verify_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=THREAD_CONFIG,
    )
    print(f"Q: {question}\nA: {extract_final_text(result['messages'][-1].content)}\n{'-' * 60}")

print(
    "Row 10 (cross-thread durable budget memory) cannot be tested locally — "
    "run mda dev . and re-ask the budget question in a NEW Studio thread instead."
)
